# 04 — Ensemble + class-bias tuning for macro-F1
**Kaggle settings:** CPU. **Inputs:** competition data + outputs of notebooks 01, 02, 03
(any subset works: every `val_*.csv` with a matching `test_*.csv` is used).
All tuning is done inside the validation fold with 5-fold *group* cross-validation, so the reported
numbers are not tuned on the rows they are measured on.

In [1]:
import os, glob, itertools
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression

INPUT_ROOT = os.environ.get("INPUT_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
def find(name):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    assert hits, f"{name} not found under {INPUT_ROOT}"
    return hits[0]
LABELS = ["human", "ai", "fusion"]; L2I = {l: i for i, l in enumerate(LABELS)}
cols = [f"s_{l}" for l in LABELS]; SEED = 42

folds = pd.read_csv(find("folds.csv")); va = folds[folds.fold == 0].reset_index(drop=True)
test = pd.read_csv(find("test.csv"), usecols=["id", "lang", "domain"])
sample_sub = pd.read_csv(find("sample_submission.csv"))
y = va.label.map(L2I).values

def log_softmax(s):
    s = s - s.max(1, keepdims=True); return s - np.log(np.exp(s).sum(1, keepdims=True))

V, T = {}, {}
for p in sorted(glob.glob(f"{INPUT_ROOT}/**/val_*.csv", recursive=True)):
    tag = os.path.basename(p)[4:-4]
    tp = glob.glob(f"{os.path.dirname(p)}/test_{tag}.csv")
    if not tp: continue
    V[tag] = log_softmax(va[["id"]].merge(pd.read_csv(p), on="id", how="left")[cols].values)
    T[tag] = log_softmax(test[["id"]].merge(pd.read_csv(tp[0]), on="id", how="left")[cols].values)
    assert not np.isnan(V[tag]).any() and not np.isnan(T[tag]).any(), tag
models = list(V); print("models:", models)

models: ['baseline', 'mdeberta', 'xlmr', 'ppl']


In [2]:
f1 = lambda yt, s: f1_score(yt, s.argmax(1), average="macro")
skf = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(va, y, va.group))

def fit_weights(S, yt, step=0.1):
    best = (-1, None)
    grid = [w for w in itertools.product(np.arange(0, 1 + 1e-9, step), repeat=len(S)) if abs(sum(w) - 1) < 1e-6]
    for w in grid:
        s = sum(wi * Si for wi, Si in zip(w, S)); sc = f1(yt, s)
        if sc > best[0]: best = (sc, w)
    return np.array(best[1])

def fit_bias(s, yt, grid=np.arange(-1.5, 1.51, 0.1)):
    best = (-1, None)
    for b1, b2 in itertools.product(grid, grid):
        b = np.array([0, b1, b2]); sc = f1(yt, s + b)
        if sc > best[0]: best = (sc, b)
    return best[1]

def stack_X(Sdict, meta):
    X = np.hstack([Sdict[m] for m in models])
    return np.hstack([X, pd.get_dummies(meta.domain).reindex(columns=sorted(va.domain.unique()), fill_value=0).values,
                      (meta.lang == "de-DE").values[:, None]]).astype(float)

def cv_eval(method):
    pred = np.zeros((len(y), 3))
    for tr_i, te_i in skf:
        pred[te_i] = method(tr_i, te_i)
    return f1(y, pred)

rows = [{"method": f"single: {m}", "cv_macro_f1": f1(y, V[m])} for m in models]
for m in models:
    rows.append({"method": f"{m} + class bias", "cv_macro_f1": cv_eval(
        lambda a, b, m=m: V[m][b] + fit_bias(V[m][a], y[a]))})
if len(models) > 1:
    def wavg(a, b, bias=False):
        w = fit_weights([V[m][a] for m in models], y[a])
        s_a = sum(wi * V[m][a] for wi, m in zip(w, models)); s_b = sum(wi * V[m][b] for wi, m in zip(w, models))
        return s_b + (fit_bias(s_a, y[a]) if bias else 0)
    rows.append({"method": "weighted avg", "cv_macro_f1": cv_eval(wavg)})
    rows.append({"method": "weighted avg + class bias", "cv_macro_f1": cv_eval(lambda a, b: wavg(a, b, True))})
Xs = stack_X(V, va)
rows.append({"method": "stacking LogReg (+domain,lang)", "cv_macro_f1": cv_eval(
    lambda a, b: LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced").fit(Xs[a], y[a]).predict_log_proba(Xs[b]))})
res = pd.DataFrame(rows).round(4).sort_values("cv_macro_f1", ascending=False)
print(res.to_string(index=False)); res.to_csv(f"{OUT}/ensemble_results.csv", index=False)

                        method  cv_macro_f1
stacking LogReg (+domain,lang)       0.9450
     weighted avg + class bias       0.9389
                  weighted avg       0.9363
         mdeberta + class bias       0.9296
             xlmr + class bias       0.9284
              single: mdeberta       0.9267
                  single: xlmr       0.9169
         baseline + class bias       0.8820
              single: baseline       0.8794
              ppl + class bias       0.7896
                   single: ppl       0.7885


## Fit the best method on the whole validation fold and predict test

In [3]:
best = res.iloc[0].method; print("best:", best)
if best.startswith("single: "):
    s_va, s_te = V[best[8:]], T[best[8:]]
elif best.endswith("+ class bias") and not best.startswith("weighted"):
    m = best.replace(" + class bias", ""); b = fit_bias(V[m], y); s_va, s_te = V[m] + b, T[m] + b
elif best.startswith("weighted"):
    w = fit_weights([V[m] for m in models], y); print("weights:", dict(zip(models, w)))
    s_va = sum(wi * V[m] for wi, m in zip(w, models)); s_te = sum(wi * T[m] for wi, m in zip(w, models))
    if "bias" in best:
        b = fit_bias(s_va, y); print("bias:", b); s_va, s_te = s_va + b, s_te + b
else:
    st = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced").fit(Xs, y)
    s_va, s_te = st.predict_log_proba(Xs), st.predict_log_proba(stack_X(T, test))
print("(in-sample, optimistic) val macro-F1:", round(f1(y, s_va), 4))
print(classification_report(y, s_va.argmax(1), target_names=LABELS, digits=4))
print(confusion_matrix(y, s_va.argmax(1), normalize="true").round(3))

sub = sample_sub[["id"]].merge(pd.DataFrame({"id": test.id, "label": np.array(LABELS)[s_te.argmax(1)]}), on="id", how="left")
assert len(sub) == len(sample_sub) and sub.id.is_unique and sub.label.isin(LABELS).all()
sub.to_csv(f"{OUT}/submission_ensemble.csv", index=False)
print(sub.label.value_counts(normalize=True).round(3).to_dict())

best: stacking LogReg (+domain,lang)
(in-sample, optimistic) val macro-F1: 0.9457
              precision    recall  f1-score   support

       human     0.8855    0.9472    0.9153      2995
          ai     0.9974    0.9940    0.9957      5456
      fusion     0.9512    0.9024    0.9262      3697

    accuracy                         0.9546     12148
   macro avg     0.9447    0.9479    0.9457     12148
weighted avg     0.9558    0.9546    0.9547     12148

[[0.947 0.001 0.052]
 [0.003 0.994 0.003]
 [0.095 0.003 0.902]]
{'ai': 0.512, 'human': 0.264, 'fusion': 0.223}
